# Kalshi Weather Edge — 5-City GFS Ensemble Pipeline

This notebook fetches 31-member GFS ensemble forecasts from Open-Meteo and compares the resulting bracket probabilities to live Kalshi market prices for the daily high-temperature series across five U.S. cities.

**Markets covered:** KXHIGHNY, KXHIGHCHI, KXHIGHMIA, KXHIGHLAX, KXHIGHDEN

**Strategy:** When `|model_prob - market_implied_prob| >= 8%`, flag as a tradeable edge.

---

## Cell 1 — Imports & City Configuration

In [ ]:
import os
import re
import logging
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
from typing import Optional
from zoneinfo import ZoneInfo

from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("kalshi_weather")

# ---------------------------------------------------------------------------
# City configuration — timezone used for the 12:00 pm trade cutoff.
# Brackets are derived dynamically from live Kalshi tickers (no temp_min/max).
# ---------------------------------------------------------------------------
CITIES: dict[str, dict] = {
    "New York": {
        "lat": 40.7128,
        "lon": -74.0060,
        "ticker_prefix": "KXHIGHNY",
        "timezone": "America/New_York",
    },
    "Chicago": {
        "lat": 41.8781,
        "lon": -87.6298,
        "ticker_prefix": "KXHIGHCHI",
        "timezone": "America/Chicago",
    },
    "Miami": {
        "lat": 25.7617,
        "lon": -80.1918,
        "ticker_prefix": "KXHIGHMIA",
        "timezone": "America/New_York",
    },
    "Los Angeles": {
        "lat": 34.0522,
        "lon": -118.2437,
        "ticker_prefix": "KXHIGHLAX",
        "timezone": "America/Los_Angeles",
    },
    "Denver": {
        "lat": 39.7392,
        "lon": -104.9903,
        "ticker_prefix": "KXHIGHDEN",
        "timezone": "America/Denver",
    },
}

print("Imports loaded. Cities configured:")
for city, cfg in CITIES.items():
    print(f"  {city:<15} {cfg['ticker_prefix']}  ({cfg['timezone']})")

---
## Cell 2 — Open-Meteo GFS Ensemble Fetch

Open-Meteo's ensemble endpoint returns one column per ensemble member. For GFS Seamless that is:
- `temperature_2m_max` — the control (unperturbed) member
- `temperature_2m_max_member01` … `temperature_2m_max_member30` — 30 perturbed members

Total: 31 members. We normalise them into columns `member_00` through `member_30`.

In [ ]:
OPEN_METEO_ENSEMBLE_URL = "https://ensemble-api.open-meteo.com/v1/ensemble"


def fetch_ensemble_forecast(
    lat: float,
    lon: float,
    forecast_days: int = 3,
) -> Optional[pd.DataFrame]:
    """
    Fetch GFS Seamless 31-member ensemble daily maximum temperature forecasts.

    Parameters
    ----------
    lat : float
        Latitude of the target location.
    lon : float
        Longitude of the target location.
    forecast_days : int
        Number of forecast days to retrieve (default 3 — today + 2 ahead).

    Returns
    -------
    pd.DataFrame or None
        Columns: ``date`` (datetime.date) plus ``member_00`` … ``member_30``
        (temperature in °F). Returns None on API failure.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max",
        "models": "gfs_seamless",
        "temperature_unit": "fahrenheit",
        "forecast_days": forecast_days,
    }

    try:
        response = requests.get(
            OPEN_METEO_ENSEMBLE_URL,
            params=params,
            timeout=20,
        )
        response.raise_for_status()
    except requests.exceptions.Timeout:
        logger.error("Open-Meteo request timed out for lat=%.4f lon=%.4f", lat, lon)
        return None
    except requests.exceptions.HTTPError as exc:
        logger.error("Open-Meteo HTTP error: %s", exc)
        return None
    except requests.exceptions.RequestException as exc:
        logger.error("Open-Meteo request failed: %s", exc)
        return None

    payload = response.json()

    # The API returns a top-level 'daily' dict whose keys look like:
    #   'time', 'temperature_2m_max', 'temperature_2m_max_member01', ...
    daily = payload.get("daily", {})
    if not daily:
        logger.error("Open-Meteo returned empty 'daily' block for lat=%.4f lon=%.4f", lat, lon)
        return None

    dates = pd.to_datetime(daily["time"]).date

    # Collect all member columns into a unified dict.
    # Control member ('temperature_2m_max') → member_00
    # Perturbed members ('temperature_2m_max_member01' … '_member30') → member_01 … member_30
    member_data: dict[str, list] = {"date": list(dates)}

    if "temperature_2m_max" in daily:
        member_data["member_00"] = daily["temperature_2m_max"]
    else:
        logger.warning("Control member 'temperature_2m_max' missing from response.")

    for key, values in daily.items():
        m = re.search(r"temperature_2m_max_member(\d+)$", key)
        if m:
            idx = int(m.group(1))  # 1-based in the API response
            col_name = f"member_{idx:02d}"
            member_data[col_name] = values

    df = pd.DataFrame(member_data)

    # Report how many members we captured.
    n_members = len([c for c in df.columns if c.startswith("member_")])
    logger.info(
        "Fetched %d ensemble members over %d days for (%.4f, %.4f).",
        n_members, len(df), lat, lon,
    )

    return df


# Smoke-test with New York — comment out if you want to preserve API quota.
print("Testing Open-Meteo fetch for New York...")
_test_df = fetch_ensemble_forecast(lat=40.7128, lon=-74.0060, forecast_days=3)
if _test_df is not None:
    member_cols = [c for c in _test_df.columns if c.startswith("member_")]
    print(f"  Rows: {len(_test_df)}, Member columns: {len(member_cols)}")
    display(_test_df.head())
else:
    print("  Fetch returned None — check connectivity or API status.")

---
## Cell 3 — Bracket Probability Calculator

Kalshi high-temp markets use 2°F brackets. Given 31 ensemble members, the model probability for a bracket is simply the fraction of members whose forecast falls inside that range.

Bracket type conventions (matching Kalshi ticker suffixes):
- `"between"` — low <= temp < high (e.g., B72.5 covers 72–74°F)
- `"above"` — temp >= low (e.g., T74)
- `"below"` — temp < high (e.g., B66 as the bottom bucket)

In [ ]:
def build_brackets(temp_min: int, temp_max: int, step: int = 2) -> list[dict]:
    """
    Generate bracket definitions for a fixed temp range (used for diagnostics only).
    In the live pipeline, brackets are derived dynamically from Kalshi tickers via
    parse_bracket_label() so they always match exactly what is open.
    """
    brackets = []
    brackets.append({"label": f"B{temp_min}", "high": temp_min, "type": "below"})
    for low in range(temp_min, temp_max, step):
        high = low + step
        brackets.append({"label": f"B{low}.5", "low": low, "high": high, "type": "between"})
    brackets.append({"label": f"T{temp_max}", "low": temp_max, "type": "above"})
    return brackets


def parse_bracket_label(label: str) -> Optional[dict]:
    """
    Convert a live Kalshi bracket label string into a bracket definition dict
    compatible with compute_bracket_probabilities.

    This replaces build_brackets() in the live pipeline — brackets are derived
    directly from whatever Kalshi tickers are open, so the label positions always
    match regardless of whether Kalshi uses odd- or even-centered bins.

    Conventions
    -----------
    T{N}    → above bracket : temp >= N        (e.g. T73  → temps >= 73°F)
    B{N}.5  → between bracket: [N-0.5, N+1.5) = [floor(N), floor(N)+2)
               (e.g. B72.5 → low=72, high=74 ; B73.5 → low=73, high=75)
    B{N}    → below bracket : temp < N         (e.g. B60  → temps < 60°F)
    """
    label = label.upper().strip()
    try:
        if label.startswith("T"):
            low = float(label[1:])
            return {"label": label, "low": low, "type": "above"}
        elif label.startswith("B"):
            val = float(label[1:])
            if "." in label[1:]:          # interior bracket  e.g. B72.5
                low = val - 0.5           # B72.5 → low=72.0
                high = low + 2.0          #       → high=74.0
                return {"label": label, "low": low, "high": high, "type": "between"}
            else:                          # bottom bracket    e.g. B60
                return {"label": label, "high": val, "type": "below"}
    except (ValueError, TypeError):
        pass
    return None


def compute_bracket_probabilities(
    ensemble_df: pd.DataFrame,
    target_date: date,
    brackets: list[dict],
) -> pd.DataFrame:
    """
    Compute GFS ensemble probability for each temperature bracket on a given date.

    Parameters
    ----------
    ensemble_df : pd.DataFrame
        Output of ``fetch_ensemble_forecast``.
    target_date : datetime.date
        The calendar date to evaluate.
    brackets : list[dict]
        Bracket definitions — from ``build_brackets`` or ``parse_bracket_label``.

    Returns
    -------
    pd.DataFrame
        Columns: ``bracket_label``, ``model_probability`` (0.0–1.0).
    """
    row = ensemble_df[ensemble_df["date"] == target_date]
    if row.empty:
        logger.warning("Target date %s not found in ensemble data.", target_date)
        return pd.DataFrame(columns=["bracket_label", "model_probability"])

    member_cols = [c for c in ensemble_df.columns if c.startswith("member_")]
    temps = row[member_cols].values.flatten().astype(float)
    temps = temps[~np.isnan(temps)]
    n_members = len(temps)

    if n_members == 0:
        logger.warning("All ensemble members are NaN for %s.", target_date)
        return pd.DataFrame(columns=["bracket_label", "model_probability"])

    results = []
    for bracket in brackets:
        btype = bracket["type"]
        label = bracket["label"]

        if btype == "below":
            count = int(np.sum(temps < bracket["high"]))
        elif btype == "above":
            count = int(np.sum(temps >= bracket["low"]))
        elif btype == "between":
            count = int(np.sum((temps >= bracket["low"]) & (temps < bracket["high"])))
        else:
            logger.error("Unknown bracket type '%s' for label %s.", btype, label)
            count = 0

        results.append({
            "bracket_label": label,
            "model_probability": round(count / n_members, 4),
        })

    return pd.DataFrame(results)


# --- Verification ---
print("parse_bracket_label smoke tests:")
assert parse_bracket_label("B72.5") == {"label": "B72.5", "low": 72.0, "high": 74.0, "type": "between"}
assert parse_bracket_label("B73.5") == {"label": "B73.5", "low": 73.0, "high": 75.0, "type": "between"}
assert parse_bracket_label("T73")   == {"label": "T73",   "low": 73.0,               "type": "above"}
assert parse_bracket_label("B60")   == {"label": "B60",   "high": 60.0,              "type": "below"}
print("  All assertions passed.")

if _test_df is not None and not _test_df.empty:
    _tomorrow = date.today() + timedelta(days=1)
    _ny_brackets = build_brackets(20, 100)
    _probs = compute_bracket_probabilities(_test_df, _tomorrow, _ny_brackets)
    _nonzero = _probs[_probs["model_probability"] > 0]
    _total = _probs["model_probability"].sum()
    print(f"\nSample bracket probabilities for NY on {_tomorrow} (total={_total:.4f}):")
    display(_nonzero.reset_index(drop=True))
    assert abs(_total - 1.0) < 1e-9, f"Probabilities do not sum to 1: {_total}"
    print("Probability sum check: PASSED")

---
## Cell 4 — Kalshi Price Fetcher

Uses `pykalshi` to pull live market prices for a given series ticker prefix.
Credentials are loaded from the `.env` file via `python-dotenv`.

Price fields returned by the API (`yes_ask_dollars`, etc.) are cent-denominated strings
(e.g., `"0.34"` means 34 cents = 34%). We convert them to floats in `[0, 1]` for the
market-implied probability.

In [ ]:
# Load Kalshi credentials from .env
load_dotenv()
KALSHI_API_KEY_ID = os.getenv("KALSHI_API_KEY_ID")
KALSHI_PRIVATE_KEY_PATH = os.getenv("KALSHI_PRIVATE_KEY_PATH")

if not KALSHI_API_KEY_ID:
    logger.warning("KALSHI_API_KEY_ID not set in .env — Kalshi fetches will fail.")
if not KALSHI_PRIVATE_KEY_PATH:
    logger.warning("KALSHI_PRIVATE_KEY_PATH not set in .env — Kalshi fetches will fail.")


def _make_kalshi_client():
    """
    Build and return a KalshiClient using credentials from environment variables.
    Raises RuntimeError if credentials are missing.
    """
    from pykalshi import KalshiClient

    key_id = os.getenv("KALSHI_API_KEY_ID")
    key_path = os.getenv("KALSHI_PRIVATE_KEY_PATH")

    if not key_id or not key_path:
        raise RuntimeError(
            "Missing Kalshi credentials. Set KALSHI_API_KEY_ID and "
            "KALSHI_PRIVATE_KEY_PATH in your .env file."
        )

    return KalshiClient(
        api_key_id=key_id,
        private_key_path=key_path,
    )


def _parse_price(raw: Optional[str]) -> Optional[float]:
    """
    Convert a Kalshi price string to a float probability in [0, 1].

    Kalshi returns dollar-denominated prices as strings where 1.00 == $1 == 100 cents
    (i.e., 100% probability). A value of "0.34" means 34 cents = 34% implied probability.
    Returns None if parsing fails.
    """
    if raw is None:
        return None
    try:
        return float(raw)  # already in [0,1] as a dollar value
    except (ValueError, TypeError):
        return None


def fetch_kalshi_prices(ticker_prefix: str) -> pd.DataFrame:
    """
    Fetch live Kalshi market prices for all active markets in a series.

    Parameters
    ----------
    ticker_prefix : str
        Series ticker, e.g. ``"KXHIGHLAX"``.

    Returns
    -------
    pd.DataFrame
        Columns: ticker, yes_ask, yes_bid, no_ask, no_bid, market_implied_prob.
        Empty DataFrame if the API call fails or no markets are found.
    """
    from pykalshi import MarketStatus

    try:
        client = _make_kalshi_client()
    except RuntimeError as exc:
        logger.error("Kalshi client init failed: %s", exc)
        return pd.DataFrame(
            columns=["ticker", "yes_ask", "yes_bid", "no_ask", "no_bid", "market_implied_prob"]
        )

    try:
        # get_markets accepts series_ticker to scope to one series.
        # fetch_all=True pages through all results automatically.
        # status must be a MarketStatus enum, not a plain string.
        markets = client.get_markets(
            series_ticker=ticker_prefix,
            status=MarketStatus.OPEN,
            fetch_all=True,
        )
    except Exception as exc:
        logger.error("Kalshi get_markets failed for %s: %s", ticker_prefix, exc)
        return pd.DataFrame(
            columns=["ticker", "yes_ask", "yes_bid", "no_ask", "no_bid", "market_implied_prob"]
        )

    if not markets:
        logger.warning("No open markets found for series %s.", ticker_prefix)
        return pd.DataFrame(
            columns=["ticker", "yes_ask", "yes_bid", "no_ask", "no_bid", "market_implied_prob"]
        )

    rows = []
    for mkt in markets:
        yes_ask = _parse_price(mkt.yes_ask_dollars)
        yes_bid = _parse_price(mkt.yes_bid_dollars)
        no_ask = _parse_price(mkt.no_ask_dollars)
        no_bid = _parse_price(mkt.no_bid_dollars)

        # Market-implied probability: mid of yes bid/ask is most robust,
        # but fall back to yes_ask alone if bid is missing.
        if yes_ask is not None and yes_bid is not None:
            market_implied_prob = (yes_ask + yes_bid) / 2.0
        elif yes_ask is not None:
            market_implied_prob = yes_ask
        elif yes_bid is not None:
            market_implied_prob = yes_bid
        else:
            market_implied_prob = None

        rows.append({
            "ticker": mkt.ticker,
            "yes_ask": yes_ask,
            "yes_bid": yes_bid,
            "no_ask": no_ask,
            "no_bid": no_bid,
            "market_implied_prob": market_implied_prob,
        })

    df = pd.DataFrame(rows)
    logger.info("Fetched %d markets for %s.", len(df), ticker_prefix)
    return df


# Smoke-test
print("Testing Kalshi price fetch for KXHIGHLAX...")
_lax_prices = fetch_kalshi_prices("KXHIGHLAX")
if not _lax_prices.empty:
    print(f"  Found {len(_lax_prices)} open markets.")
    display(_lax_prices.head())
else:
    print("  No markets returned (check credentials or market availability).")

---
## Cell 5 — Edge Calculator (Core Signal Logic)

This is the heart of the pipeline. For each city it:

1. Fetches the 31-member GFS ensemble for tomorrow.
2. Computes bracket probabilities from the ensemble.
3. Fetches live Kalshi prices.
4. Extracts the bracket label from each Kalshi ticker using a regex.
5. Merges on the bracket label.
6. Computes `edge = model_prob − market_implied_prob`.
7. Flags actionable edges where `|edge| >= min_edge`.

**Ticker format example:** `KXHIGHLAX-26MAR14-B72.5`
- Series:  `KXHIGHLAX`
- Date:    `26MAR14`  (YY-MMM-DD, no separator)
- Bracket: `B72.5`

In [ ]:
# Regex to extract the bracket portion from a Kalshi weather ticker.
#   KXHIGHLAX-26MAR14-B72.5  → group(1) = "B72.5"
#   KXHIGHCHI-26MAR14-T68    → group(1) = "T68"
_BRACKET_RE = re.compile(r"-\d{2}[A-Z]{3}\d{2}-(.+)$", re.IGNORECASE)


def _extract_bracket_label(ticker: str) -> Optional[str]:
    m = _BRACKET_RE.search(ticker)
    return m.group(1).upper() if m else None


def find_edges(
    city_name: str,
    ticker_prefix: str,
    lat: float,
    lon: float,
    timezone: str,
    target_date: Optional[date] = None,
    min_edge: float = 0.08,
) -> pd.DataFrame:
    """
    Identify tradeable edges between GFS ensemble probabilities and Kalshi prices.

    Filters applied
    ---------------
    - Noon cutoff  : skips city if local time >= 12:00 pm (DST-aware via zoneinfo).
    - Market floor : drops brackets with market_implied_prob < 3 % (illiquid).
    - Model ceiling: drops brackets where model_probability == 100 % (degenerate).
    - Min edge     : keeps only |model_prob - market_prob| >= min_edge.

    Brackets are derived dynamically from live Kalshi tickers so label positions
    always match regardless of odd/even bin offset Kalshi uses on any given day.
    """
    if target_date is None:
        target_date = date.today() + timedelta(days=1)

    # ── Noon cutoff (DST-aware) ──────────────────────────────────────────────
    local_now = datetime.now(ZoneInfo(timezone))
    if local_now.hour >= 12:
        logger.info(
            "%s: Past noon cutoff (%s local) — no signals generated.",
            city_name, local_now.strftime("%H:%M"),
        )
        return pd.DataFrame()

    logger.info("Processing %s (%s) for %s", city_name, ticker_prefix, target_date)

    # --- Step 1: GFS ensemble forecast ---
    ensemble_df = fetch_ensemble_forecast(lat, lon, forecast_days=3)
    if ensemble_df is None or ensemble_df.empty:
        logger.error("No ensemble data for %s — skipping.", city_name)
        return pd.DataFrame()

    # --- Step 2: Kalshi live prices (fetched FIRST so we can derive brackets) ---
    price_df = fetch_kalshi_prices(ticker_prefix)
    if price_df.empty:
        logger.warning("No Kalshi prices for %s — skipping.", ticker_prefix)
        return pd.DataFrame()

    # --- Step 3: Extract bracket labels from Kalshi tickers ---
    price_df = price_df.copy()
    price_df["bracket_label"] = price_df["ticker"].apply(_extract_bracket_label)
    price_df.dropna(subset=["bracket_label", "market_implied_prob"], inplace=True)
    price_df["bracket_label"] = price_df["bracket_label"].str.upper()

    # ── Market probability floor: drop illiquid / near-zero contracts ────────
    price_df = price_df[price_df["market_implied_prob"] >= 0.03]
    if price_df.empty:
        logger.warning("%s: All markets below 3%% threshold — skipping.", ticker_prefix)
        return pd.DataFrame()

    if price_df.empty:
        logger.warning("No parseable bracket labels for %s.", ticker_prefix)
        return pd.DataFrame()

    # --- Step 4: Build bracket definitions directly from Kalshi labels ---
    kalshi_labels = price_df["bracket_label"].unique().tolist()
    brackets = [b for b in (parse_bracket_label(lbl) for lbl in kalshi_labels) if b is not None]

    if not brackets:
        logger.error(
            "Could not parse any bracket definitions from Kalshi labels for %s: %s",
            city_name, kalshi_labels,
        )
        return pd.DataFrame()

    # --- Step 5: Compute ensemble probabilities against the actual Kalshi brackets ---
    prob_df = compute_bracket_probabilities(ensemble_df, target_date, brackets)
    if prob_df.empty:
        logger.error("No bracket probabilities computed for %s on %s.", city_name, target_date)
        return pd.DataFrame()

    prob_df["bracket_label"] = prob_df["bracket_label"].str.upper()

    # ── Model probability ceiling: drop 100%-certain predictions (degenerate) ─
    prob_df = prob_df[prob_df["model_probability"] < 1.0]
    if prob_df.empty:
        logger.warning("%s: All brackets at 100%% model probability — skipping.", city_name)
        return pd.DataFrame()

    # --- Step 6: Merge — labels guaranteed to match (both derived from Kalshi) ---
    merged = price_df.merge(prob_df, on="bracket_label", how="inner")
    if merged.empty:
        logger.warning(
            "Merge still empty for %s — Kalshi labels: %s",
            city_name, kalshi_labels,
        )
        return pd.DataFrame()

    # --- Step 7: Compute edge ---
    merged["edge"] = merged["model_probability"] - merged["market_implied_prob"]
    merged["abs_edge"] = merged["edge"].abs()
    merged["recommended_side"] = np.where(merged["edge"] > 0, "YES", "NO")

    # --- Step 8: Filter by minimum edge ---
    signals = merged[merged["abs_edge"] >= min_edge].copy()
    signals["city"] = city_name

    output_cols = [
        "city", "ticker", "bracket_label",
        "model_probability", "market_implied_prob",
        "edge", "recommended_side",
    ]
    signals = signals[output_cols].sort_values("edge", key=abs, ascending=False)

    logger.info(
        "%s: %d signals found above %.0f%% edge threshold.",
        city_name, len(signals), min_edge * 100,
    )
    return signals.reset_index(drop=True)


# Unit tests for the ticker-parsing helper
assert _extract_bracket_label("KXHIGHLAX-26MAR14-B72.5") == "B72.5"
assert _extract_bracket_label("KXHIGHCHI-26MAR14-T68")   == "T68"
assert _extract_bracket_label("KXHIGHNY-26MAR14-B20")    == "B20"
assert _extract_bracket_label("KXHIGHMIA-26MAR14-B80.5") == "B80.5"
assert _extract_bracket_label("KXHIGHDEN-26MAR14-B60")   == "B60"
print("All ticker-parsing assertions passed.")

---
## Cell 6 — Run All 5 Cities

Loops through every city, calls `find_edges`, and assembles a single opportunity table sorted by absolute edge descending.

In [ ]:
TARGET_DATE = date.today() + timedelta(days=1)  # tomorrow's high
MIN_EDGE = 0.08  # 8% minimum edge

print(f"Running 5-city sweep for target date: {TARGET_DATE}")
print(f"Minimum edge threshold: {MIN_EDGE:.0%}")
print("-" * 60)

all_signals: list[pd.DataFrame] = []

for city_name, cfg in CITIES.items():
    signals = find_edges(
        city_name=city_name,
        ticker_prefix=cfg["ticker_prefix"],
        lat=cfg["lat"],
        lon=cfg["lon"],
        timezone=cfg["timezone"],
        target_date=TARGET_DATE,
        min_edge=MIN_EDGE,
    )
    all_signals.append(signals)

# Concatenate and sort
non_empty = [df for df in all_signals if not df.empty]

if non_empty:
    opportunity_table = (
        pd.concat(non_empty, ignore_index=True)
        .assign(abs_edge=lambda d: d["edge"].abs())
        .sort_values("abs_edge", ascending=False)
        .drop(columns="abs_edge")
        .reset_index(drop=True)
    )

    display_table = opportunity_table.copy()
    for col in ["model_probability", "market_implied_prob", "edge"]:
        display_table[col] = display_table[col].map("{:+.1%}".format if col == "edge" else "{:.1%}".format)

    print(f"\nTotal signals found: {len(opportunity_table)}")
    display(display_table)
else:
    opportunity_table = pd.DataFrame()
    print("\nNo signals exceeded the edge threshold across all 5 cities.")
    print("This is expected if markets are already efficiently priced, no open markets were found,")
    print("or all cities are past the 12:00 pm local trade cutoff.")

---
## Cell 7 — Daily Signal Summary

Prints a formatted console-style summary suitable for copy-pasting into a Slack message
or daily trade log.

In [ ]:
def print_signal_summary(
    signals: pd.DataFrame,
    target_date: date,
    min_edge: float = 0.08,
    top_n: int = 10,
) -> None:
    """
    Print a formatted daily signal summary to stdout.

    Parameters
    ----------
    signals : pd.DataFrame
        Output of the 5-city concatenation step.
    target_date : datetime.date
        The forecast date.
    min_edge : float
        Edge threshold used (for display only).
    top_n : int
        Maximum number of opportunities to print.
    """
    header = f"KALSHI WEATHER EDGE SIGNALS — {target_date.strftime('%Y-%m-%d')}"
    separator = "=" * max(len(header), 70)

    print(separator)
    print(f"  {header}")
    print(separator)

    if signals.empty:
        n_signals = 0
        print(f"  Found {n_signals} total signals across 5 cities (min edge: {min_edge:.0%})")
        print("  No actionable opportunities at current threshold.")
        print(separator)
        return

    n_signals = len(signals)
    n_cities_with_signals = signals["city"].nunique()

    print(f"  Found {n_signals} total signals across {n_cities_with_signals}/5 cities (min edge: {min_edge:.0%})")
    print()
    print("  TOP OPPORTUNITIES:")
    print()

    # Column header
    print(
        f"  {'City':<15} "
        f"{'Ticker':<32} "
        f"{'Model%':>7} "
        f"{'Market%':>8} "
        f"{'Edge':>7} "
        f"{'Side':>4}"
    )
    print("  " + "-" * 78)

    top = signals.head(top_n)
    for _, row in top.iterrows():
        edge_sign = "+" if row["edge"] > 0 else ""
        print(
            f"  {row['city']:<15} "
            f"{row['ticker']:<32} "
            f"{row['model_probability']:>6.1%} "
            f"{row['market_implied_prob']:>8.1%} "
            f"{edge_sign}{row['edge']:>6.1%} "
            f"{row['recommended_side']:>4}"
        )

    if n_signals > top_n:
        print(f"  ... and {n_signals - top_n} more signals not shown.")

    print()
    print("  NOTE: Edge = Model Probability - Market Implied Probability.")
    print("        Positive edge → buy YES. Negative edge → buy NO.")
    print("        Always verify current Kalshi orderbook before entering a position.")
    print(separator)


print_signal_summary(
    signals=opportunity_table,
    target_date=TARGET_DATE,
    min_edge=MIN_EDGE,
    top_n=10,
)

---
## Cell 8 — Diagnostics: Ensemble Distribution Inspection

Optional cell to drill into the raw ensemble distribution for any single city and date.
Useful for understanding model spread and sanity-checking bracket assignments.

In [ ]:
def inspect_city_ensemble(
    city_name: str,
    target_date: Optional[date] = None,
) -> None:
    """
    Print ensemble spread statistics and all non-zero bracket probabilities for a city.
    Brackets are derived from live Kalshi tickers (same as find_edges).

    Parameters
    ----------
    city_name : str
        Must match a key in CITIES.
    target_date : datetime.date or None
        Defaults to tomorrow.
    """
    if city_name not in CITIES:
        print(f"Unknown city '{city_name}'. Valid: {list(CITIES.keys())}")
        return

    cfg = CITIES[city_name]
    if target_date is None:
        target_date = date.today() + timedelta(days=1)

    print(f"Ensemble diagnostics for {city_name} on {target_date}")
    print("-" * 50)

    ens = fetch_ensemble_forecast(cfg["lat"], cfg["lon"], forecast_days=3)
    if ens is None or ens.empty:
        print("Failed to retrieve ensemble data.")
        return

    row = ens[ens["date"] == target_date]
    if row.empty:
        print(f"Date {target_date} not in forecast window. Available: {list(ens['date'])}")
        return

    member_cols = [c for c in ens.columns if c.startswith("member_")]
    temps = row[member_cols].values.flatten().astype(float)
    temps = temps[~np.isnan(temps)]

    print(f"  Members available : {len(temps)} / 31")
    print(f"  Min               : {temps.min():.1f}°F")
    print(f"  Max               : {temps.max():.1f}°F")
    print(f"  Mean              : {temps.mean():.1f}°F")
    print(f"  Median            : {np.median(temps):.1f}°F")
    print(f"  Std dev           : {temps.std():.1f}°F")
    print(f"  10th / 90th pct   : {np.percentile(temps, 10):.1f} / {np.percentile(temps, 90):.1f}°F")
    print()

    # Derive brackets from live Kalshi tickers (mirrors find_edges logic)
    price_df = fetch_kalshi_prices(cfg["ticker_prefix"])
    if price_df.empty:
        print("  No open Kalshi markets — cannot show bracket probabilities.")
        return

    price_df["bracket_label"] = price_df["ticker"].apply(_extract_bracket_label)
    price_df.dropna(subset=["bracket_label"], inplace=True)
    price_df["bracket_label"] = price_df["bracket_label"].str.upper()

    kalshi_labels = price_df["bracket_label"].unique().tolist()
    brackets = [b for b in (parse_bracket_label(lbl) for lbl in kalshi_labels) if b is not None]

    if not brackets:
        print("  Could not parse bracket definitions from Kalshi labels.")
        return

    prob_df = compute_bracket_probabilities(ens, target_date, brackets)
    nonzero = prob_df[prob_df["model_probability"] > 0]
    print(f"  Non-zero brackets ({len(nonzero)}):")
    for _, r in nonzero.iterrows():
        bar = "#" * int(r["model_probability"] * 40)
        print(f"    {r['bracket_label']:<10} {r['model_probability']:>6.1%}  {bar}")


# Run diagnostics for Los Angeles as an example
inspect_city_ensemble("Los Angeles", TARGET_DATE)